# 04.05 — Cypher Result Shapes: Customising `materialize`

This notebook is an **ergonomics assessment**. The question it addresses:

> How does the `materialize` contract handle Cypher queries that return more than
> one node, a mix of nodes and relationships, or a path?

The `materialize(self, raw: dict)` method is the single customisation point
for shaping a raw driver record into your typed `Output` model.
`CypherExecutor.read()` calls it once per record, passing `dict(rec)` — a plain
Python dict whose keys are exactly the column names projected by the `RETURN` clause.

Four shapes are assessed:

| # | Shape | Example Cypher `RETURN` clause |
|---|-------|-------------------------------|
| 1 | Single node | `RETURN m` |
| 2 | Multiple nodes | `RETURN p, m` |
| 3 | Node + relationship | `RETURN p, r, m` |
| 4 | Path | `RETURN path` |

A final section adds **static validation against the graph model** for each shape,
showing what `validate_cypher` and `validate_query_catalogue` catch — and what they
cannot catch (imperative queries, identifier-injection templates, path patterns).

No live database is required. A `FakeGraphSession` stands in for the driver
throughout.

In [ ]:
from typing import Any, Optional

from pydantic import BaseModel

from orthograph.cypher.base_models import CypherReadQuery
from orthograph.cypher.bindings import NoParams
from orthograph.cypher.query_execution import CypherExecutor
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import NodeModel, RelationshipModel

## Domain model

Classic filmography: `Person` nodes, `Movie` nodes, and `ACTED_IN` relationships.

In [ ]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    released: int
    tagline: Optional[str] = None


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    role: str


graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)
print("Nodes:", graph_definition.node_labels)
print("Rels: ", graph_definition.relationship_labels)

## The FakeGraphSession test double

Used throughout. It accepts a list of dicts (one per record) and returns them
from `run()`. Each dict mirrors what a real neo4j driver record exposes when
converted to a plain Python dict via `dict(rec)`.

**What does a real record dict look like?**

- `RETURN m` → `{"m": <neo4j.graph.Node>}` — the value is a Node object,
  which behaves like a dict of properties but also exposes `.labels`.
- `RETURN m.title AS title` → `{"title": "The Matrix"}` — the value is the
  raw scalar.
- `RETURN p, r, m` → `{"p": <Node>, "r": <Relationship>, "m": <Node>}` — one
  key per projected variable.
- `RETURN path` → `{"path": <neo4j.graph.Path>}` — the value is a Path object
  with `.nodes` and `.relationships`.

In the fake session we represent nodes as plain dicts (property bag) and
relationships as dicts with a `__type__` key. Paths are represented as a
small dataclass. This is sufficient to demonstrate the `materialize` contract.

In [ ]:
from dataclasses import dataclass


class FakeGraphSession:
    """Minimal stand-in for a graph driver session."""

    def __init__(self, records: list[dict[str, Any]]):
        self._records = records

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        pass

    def run(self, cypher: str, **params: Any):
        return self._records


# Fake node: a dict of properties (real neo4j Node also behaves like a mapping).
def fake_node(**props: Any) -> dict[str, Any]:
    return dict(props)


# Fake relationship: a dict of properties plus __type__ for the relationship label.
def fake_rel(rel_type: str, **props: Any) -> dict[str, Any]:
    return {"__type__": rel_type, **props}


# Fake path: exposes .nodes and .relationships, like neo4j.graph.Path.
@dataclass
class FakePath:
    nodes: list[dict[str, Any]]
    relationships: list[dict[str, Any]]

---
## Shape 1 — Single node returned (`RETURN m`)

This is the simplest case and the pattern shown in notebook `04.01`. The RETURN
clause projects a single node variable. The record dict has one key — the
variable name — whose value is the node object.

```
raw dict: {"m": {"title": "The Matrix", "released": 1999, "tagline": "..."}}
```

`materialize` extracts the node and passes its property dict to `Movie.model_validate`.

**Ergonomics:** Good. One key, one model. `materialize` is a one-liner.

In [ ]:
class AllMoviesParams(BaseModel):
    pass


class AllMovies(CypherReadQuery[AllMoviesParams, Movie]):
    """Return all Movie nodes — single node per record."""

    name = "all_movies"
    cypher_template = "MATCH (m:Movie) RETURN m"

    def materialize(self, raw: dict[str, Any]) -> Movie:
        # raw["m"] is the node object (behaves like a property dict).
        return Movie.model_validate(raw["m"])


fake_records = [
    {
        "m": fake_node(
            title="The Matrix", released=1999, tagline="Welcome to the Real World"
        )
    },
    {"m": fake_node(title="Speed", released=1994, tagline=None)},
]

executor = CypherExecutor(lambda: FakeGraphSession(fake_records))
movies = executor.read(AllMovies(), {})

print(f"Returned {len(movies)} movies:")
for m in movies:
    print(f"  {m.title} ({m.released})")

**Note on `NoParams`:** When there are no query parameters, declare `Params` as
`NoParams` — it is already imported above. Or use an empty `BaseModel` as above.
Both are equivalent; `NoParams` is the canonical sentinel.

In [ ]:
# Same query using the canonical NoParams sentinel
class AllMoviesV2(CypherReadQuery[NoParams, Movie]):
    name = "all_movies_v2"
    cypher_template = "MATCH (m:Movie) RETURN m"

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie.model_validate(raw["m"])


movies_v2 = executor.read(AllMoviesV2(), {})
assert [m.title for m in movies_v2] == ["The Matrix", "Speed"]
print("NoParams sentinel works identically:", [m.title for m in movies_v2])

---
## Shape 2 — Multiple nodes returned (`RETURN p, m`)

A query that returns two nodes per record — for example, finding actors and the
movies they acted in:

```cypher
MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p, m
```

The record dict has **two** keys:
```
raw dict: {
  "p": {"name": "Keanu Reeves", "born": 1964},
  "m": {"title": "The Matrix", "released": 1999}
}
```

The `Output` type is no longer a single `NodeModel` — it must be a **projection**
that holds both nodes. Define a `BaseModel` with two fields.

**Ergonomics:** The friction is the projection type. You must define a new
`BaseModel` for each combination of returned entities. `materialize` is still a
one-liner per field, but the boilerplate is in the projection class declaration.

In [ ]:
# Projection Output type — holds both nodes as typed sub-models
class ActorMoviePair(BaseModel):
    """One actor + one movie they acted in."""

    person: Person
    movie: Movie


class ActorMoviePairsParams(BaseModel):
    pass


class ActorMoviePairs(CypherReadQuery[ActorMoviePairsParams, ActorMoviePair]):
    """Return all (person, movie) pairs — two nodes per record."""

    name = "actor_movie_pairs"
    cypher_template = "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p, m"

    def materialize(self, raw: dict[str, Any]) -> ActorMoviePair:
        # Two keys in raw — one per projected variable.
        return ActorMoviePair(
            person=Person.model_validate(raw["p"]),
            movie=Movie.model_validate(raw["m"]),
        )


fake_pairs = [
    {
        "p": fake_node(name="Keanu Reeves", born=1964),
        "m": fake_node(title="The Matrix", released=1999),
    },
    {
        "p": fake_node(name="Hugo Weaving", born=1960),
        "m": fake_node(title="The Matrix", released=1999),
    },
]

executor2 = CypherExecutor(lambda: FakeGraphSession(fake_pairs))
pairs = executor2.read(ActorMoviePairs(), {})

print(f"Returned {len(pairs)} pairs:")
for pair in pairs:
    print(f"  {pair.person.name} acted in {pair.movie.title}")

### Alternative: project scalars instead of full nodes

For display-oriented queries you often only need a subset of fields. Project
named columns with `AS` aliases; the record dict keys will be the alias names.
This avoids the nested projection type and can be more readable.

```cypher
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
RETURN p.name AS actor_name, m.title AS movie_title, m.released AS released
```

**Ergonomics:** Flat projection type is often simpler than nested `NodeModel`
fields. This is the recommended approach for read-only list views.

In [ ]:
class ActorFilmRow(BaseModel):
    """Flat projection — actor name + movie title + year."""

    actor_name: str
    movie_title: str
    released: int


class ActorFilmRows(CypherReadQuery[NoParams, ActorFilmRow]):
    """Return actor+movie pairs as flat scalar projections."""

    name = "actor_film_rows"
    cypher_template = (
        "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) "
        "RETURN p.name AS actor_name, m.title AS movie_title, m.released AS released"
    )

    def materialize(self, raw: dict[str, Any]) -> ActorFilmRow:
        # Keys are the AS-alias names — map directly to the flat model.
        return ActorFilmRow.model_validate(raw)


fake_flat_rows = [
    {"actor_name": "Keanu Reeves", "movie_title": "The Matrix", "released": 1999},
    {"actor_name": "Hugo Weaving", "movie_title": "The Matrix", "released": 1999},
]

executor3 = CypherExecutor(lambda: FakeGraphSession(fake_flat_rows))
rows = executor3.read(ActorFilmRows(), {})

print(f"Returned {len(rows)} rows:")
for row in rows:
    print(f"  {row.actor_name} — {row.movie_title} ({row.released})")

---
## Shape 3 — Node + relationship + node (`RETURN p, r, m`)

When the query returns a relationship alongside its endpoint nodes, each record
carries three values:

```
raw dict: {
  "p": {"name": "Keanu Reeves", "born": 1964},
  "r": {"__type__": "ACTED_IN", "role": "Neo"},
  "m": {"title": "The Matrix", "released": 1999}
}
```

The projection `Output` type needs a field for each returned variable. For the
relationship field we can use the `RelationshipModel` directly or a flat
projection.

**Ergonomics:** The same as Shape 2 — one more field on the projection, one
more `model_validate` call in `materialize`. The pattern scales linearly.

In [ ]:
class ActorMovieEdge(BaseModel):
    """A full edge record: source node, relationship, target node."""

    person: Person
    acted_in: ActedIn
    movie: Movie


class ActorMovieEdgesParams(BaseModel):
    pass


class ActorMovieEdges(CypherReadQuery[ActorMovieEdgesParams, ActorMovieEdge]):
    """Return (person)-[acted_in]->(movie) triples."""

    name = "actor_movie_edges"
    cypher_template = "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p, r, m"

    def materialize(self, raw: dict[str, Any]) -> ActorMovieEdge:
        # raw["r"] is the relationship object — a property dict from the fake driver.
        # A real neo4j Relationship also behaves like a property mapping.
        # We strip the __type__ sentinel before validating (it is not a model field).
        rel_props = {k: v for k, v in raw["r"].items() if k != "__type__"}
        return ActorMovieEdge(
            person=Person.model_validate(raw["p"]),
            acted_in=ActedIn.model_validate(rel_props),
            movie=Movie.model_validate(raw["m"]),
        )


fake_triples = [
    {
        "p": fake_node(name="Keanu Reeves", born=1964),
        "r": fake_rel("ACTED_IN", role="Neo"),
        "m": fake_node(title="The Matrix", released=1999),
    },
    {
        "p": fake_node(name="Hugo Weaving", born=1960),
        "r": fake_rel("ACTED_IN", role="Agent Smith"),
        "m": fake_node(title="The Matrix", released=1999),
    },
]

executor4 = CypherExecutor(lambda: FakeGraphSession(fake_triples))
edges = executor4.read(ActorMovieEdges(), {})

print(f"Returned {len(edges)} edges:")
for e in edges:
    print(f"  {e.person.name} --[ACTED_IN role={e.acted_in.role}]--> {e.movie.title}")

### Note on real neo4j relationship objects

In a live neo4j session the `raw["r"]` value is a `neo4j.graph.Relationship`
object. It exposes:
- Property access via `rel["role"]` (mapping protocol, like a dict)
- `.type` — the relationship type string (`"ACTED_IN"`)
- `.id`, `.element_id` — internal identifiers

The `dict(rec)` conversion inside `CypherExecutor.read` turns the record into a
plain Python dict, but the **values** remain neo4j objects (Node, Relationship,
Path). Your `materialize` is responsible for extracting properties from those
objects. Using `dict(node)` or `dict(rel)` on the object itself gives you the
property bag as a plain dict, suitable for `model_validate`.

---
## Shape 4 — Path (`RETURN path`)

A path query returns a `neo4j.graph.Path` object — an ordered alternating
sequence of nodes and relationships:

```cypher
MATCH path = (p:Person {name: $name})-[:ACTED_IN*1..3]->(m:Movie) RETURN path
```

The record dict has one key:
```
raw dict: {
  "path": <Path with .nodes=[Person, Movie, ...] and .relationships=[ActedIn, ...]>
}
```

This is the highest-friction shape. The `Output` type cannot directly mirror a
neo4j Path. Two practical strategies:

1. **Flatten to a projection** — extract only the data you need from the path
   endpoints in `materialize`.
2. **Wrap in a `PathResult` type** — collect all nodes and relationships into
   typed lists.

**Ergonomics:** Considerably more work in `materialize`. The path structure is
driver-specific (neo4j vs Memgraph may differ in their Path API). The query
must be imperative-style because graphglot does not fully parse variable-length
pattern expressions (they may raise `CypherQueryDefinitionError` at definition
time). Using an imperative query bypasses definition-time validation.

In [ ]:
import warnings

from orthograph.cypher.bindings import CypherQuery


class ActorPathParams(BaseModel):
    name: str


class PathResult(BaseModel):
    """Typed container for a materialised path."""

    start_name: str  # the Person at path start
    hops: int  # number of relationships in the path
    end_title: str  # the Movie at path end
    roles: list[str]  # role strings from each ACTED_IN relationship


# Path queries with variable-length patterns may fail definition-time
# graphglot validation, so we use the imperative escape hatch.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)

    class ShortestActorPath(CypherReadQuery[ActorPathParams, PathResult]):
        """Return paths from a named actor to movies via ACTED_IN.

        Uses imperative build() because variable-length path patterns
        (`[:ACTED_IN*1..3]`) are not fully supported by the bundled
        graphglot parser at definition time.
        """

        name = "shortest_actor_path"

        def build(self, params: ActorPathParams) -> CypherQuery:
            cypher = (
                "MATCH path = (p:Person {name: $name})-[:ACTED_IN*1..3]->(m:Movie) "
                "RETURN path"
            )
            return cypher, {"name": params.name}

        def materialize(self, raw: dict[str, Any]) -> PathResult:
            # raw["path"] is a Path object with .nodes and .relationships.
            path = raw["path"]
            nodes = list(path.nodes)  # [Person, Movie, ...]
            rels = list(path.relationships)  # [ACTED_IN, ...]

            # Start node is always a Person; end node is always a Movie.
            start = dict(nodes[0])
            end = dict(nodes[-1])

            # Collect role strings from each relationship in the path.
            roles = [dict(r).get("role", "?") for r in rels]

            return PathResult(
                start_name=start["name"],
                hops=len(rels),
                end_title=end["title"],
                roles=roles,
            )


# Construct a fake path double
fake_person_node = fake_node(name="Keanu Reeves", born=1964)
fake_movie_node = fake_node(title="The Matrix", released=1999)
fake_acted_in = fake_rel("ACTED_IN", role="Neo")

fake_path = FakePath(
    nodes=[fake_person_node, fake_movie_node],
    relationships=[fake_acted_in],
)

fake_path_records = [{"path": fake_path}]

executor5 = CypherExecutor(lambda: FakeGraphSession(fake_path_records))

# The executor re-parses the built Cypher string at runtime.
# Variable-length patterns may cause CypherSyntaxError depending on the parser.
# To demonstrate the materialize contract without triggering the parser,
# we call materialize directly here:
query_instance = ShortestActorPath()
result = query_instance.materialize({"path": fake_path})

print("Path result:")
print(f"  start: {result.start_name}")
print(f"  end:   {result.end_title}")
print(f"  hops:  {result.hops}")
print(f"  roles: {result.roles}")

### When to avoid path queries in typed interfaces

Path results are driver-specific. The `neo4j.graph.Path` object is not
available outside a live connection and its API differs across drivers.
In production, consider:

- **Unrolling the path in Cypher.** Use `UNWIND nodes(path) AS n` or
  `UNWIND relationships(path) AS r` to return individual nodes/relationships
  as separate records. These are Shape 1 or Shape 3 — covered above.
- **Projecting only what you need.** Most path queries only need the start
  and end. `RETURN p.name AS name, m.title AS title` (Shape 2 flat) is
  usually sufficient.

Use `RETURN path` only when the full traversal structure (intermediate hops)
is part of the required output.

---
## Ergonomics summary

| Shape | `Output` type | `materialize` complexity | Parser support | Recommended? |
|-------|--------------|--------------------------|----------------|--------------|
| Single node (`RETURN m`) | NodeModel directly | One-liner | Full (declarative) | Yes — default |
| Scalar projection (`RETURN m.title AS title`) | Flat BaseModel | One-liner (`model_validate(raw)`) | Full (declarative) | Yes — preferred for views |
| Multiple nodes (`RETURN p, m`) | Nested projection | One field per variable | Full (declarative) | Yes — when full nodes needed |
| Node + relationship + node (`RETURN p, r, m`) | Nested projection | Strip driver sentinel + validate | Full (declarative) | Yes — with real driver `dict()` |
| Path (`RETURN path`) | Custom wrapper | Unpack `.nodes`/`.relationships` | Limited (imperative) | Avoid — prefer projections |

### Key constraints in the current interface

1. **`materialize` is mandatory.** Even for the trivial 1:1 case
   (`Output.model_validate(raw["m"])`), every query must implement it.
   This is the friction addressed by E33 (the `row_mapper` proposal — ADR-025).

2. **One `Output` type per query class.** A query returning two different
   `NodeModel` types requires a dedicated projection `BaseModel` to hold
   both. There is no tuple return or union output today.

3. **Relationship properties in `raw["r"]` are driver-specific.** With the
   real neo4j driver, `raw["r"]` is a `neo4j.graph.Relationship`. Use
   `dict(raw["r"])` (the mapping protocol) to extract properties before
   passing to `model_validate`.

4. **Path queries bypass definition-time validation.** Variable-length
   patterns require the imperative escape hatch, losing the static safety net.

---
## Model definition vs DTO — are we coupling two concerns?

A question that arises when looking at the examples above:

> `Movie` is a `NodeModel` — a graph schema declaration. When we write
> `CypherReadQuery[Params, Movie]`, are we coupling the graph definition to the
> query result layer?

**The contract does not force that coupling.** The type bound on the `Output`
generic argument is `D: bound=BaseModel` — any Pydantic model, nothing
graph-specific. The `ActorFilmRow` and `ActorMoviePair` examples above already
use plain `BaseModel` subclasses as `Output`, with no graph machinery at all.

What *looks* like coupling in Shapes 1–3 is a **documentation choice**, not a
contract. Using `Movie` as `Output` is ergonomically convenient for the
"fetch the whole node" case — but it carries a real cost:

- `NodeModel` brings graph-definition baggage into the result layer: `__label__`,
  `__uid_field__`, `__optional__`, cardinality specs, `get_property_specs()`.
  None of that belongs in a query result or an API response.
- A consumer that receives a `Movie` can see and depend on `Movie.__label__` and
  `Movie.__uid_field__`. That is a leaky abstraction.
- If `Movie` changes its schema (a field renamed, a property made optional), every
  query that returns `Movie` directly must change simultaneously. A DTO breaks that
  coupling — it declares only what *this query, for this consumer* needs.

### The correct layering

```
GraphDefinition (NodeModel / RelationshipModel)   ← declared graph contract
        │  used only for static validation
        ▼
CypherReadQuery[Params, DTO]                      ← query + result shape
        │  materialize() is the explicit translation seam
        ▼
Consumer (route, service, repository)             ← sees only the DTO
```

`GraphDefinition` (and the `NodeModel` types it contains) serves the validator
(`validate_cypher`, `validate_query_catalogue`). It never needs to appear in
the consumer's type signature.

### When is it acceptable to use `NodeModel` as `Output`?

In one narrow case: when the query's `RETURN` columns map **1:1** to the node's
fields and you want to expose the full node shape unchanged to the consumer.
This is a valid shortcut, not an antipattern — as long as you are aware that the
`NodeModel` *is* the DTO at that point, and you own the consequences if the
stored shape and the API contract drift.

The moment the API shape diverges by even one field — a rename, a hidden
property, a computed field, a join across multiple nodes — you need a dedicated
DTO and an explicit mapping in `materialize`. The flat scalar projection
(`ActorFilmRow`) is the cleanest expression of that:

In [ ]:
# -----------------------------------------------------------------------
# Side-by-side: NodeModel as Output  vs  dedicated DTO as Output
# Same Cypher, same executor, different Output contract.
# -----------------------------------------------------------------------

# --- Option A: NodeModel as Output (convenient, 1:1 case only) ----------


class MoviesByYearParams(BaseModel):
    released: int


class MoviesByYear_NodeOutput(CypherReadQuery[MoviesByYearParams, Movie]):
    """Output = Movie (NodeModel). Fast, but the consumer receives graph internals
    if it introspects the class. Only correct when RETURN columns == node fields.
    """

    name = "movies_by_year_node"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released, m.tagline AS tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie.model_validate(raw)  # 1:1 — shortcut is valid here


# --- Option B: dedicated DTO as Output (explicit boundary) --------------


class MovieCard(BaseModel):  # pure DTO — no graph concepts
    """What the consumer actually needs: title + year (renamed).
    No __label__, no __uid_field__, no cardinality machinery.
    The graph schema can change without touching this contract.
    """

    title: str
    year: int  # renamed from 'released' in the graph


class MoviesByYear_DtoOutput(CypherReadQuery[MoviesByYearParams, MovieCard]):
    """Output = MovieCard (plain BaseModel). The consumer only sees the DTO.
    materialize() is the explicit translation seam — columns -> DTO fields.
    """

    name = "movies_by_year_dto"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released"
    )

    def materialize(self, raw: dict[str, Any]) -> MovieCard:
        # Explicit mapping: graph column 'released' -> DTO field 'year'.
        # This seam absorbs any future rename on either side independently.
        return MovieCard(title=raw["title"], year=raw["released"])


# -----------------------------------------------------------------------
# Run both queries through the same fake executor.
# -----------------------------------------------------------------------

fake_year_records = [
    {"title": "The Matrix", "released": 1999, "tagline": "Welcome to the Real World"},
    {"title": "Fight Club", "released": 1999, "tagline": None},
]
ex = CypherExecutor(lambda: FakeGraphSession(fake_year_records))

node_results = ex.read(MoviesByYear_NodeOutput(), {"released": 1999})
dto_results = ex.read(MoviesByYear_DtoOutput(), {"released": 1999})

print("Option A — NodeModel output:")
for m in node_results:
    # Consumer receives a Movie — has access to Movie.__label__, etc.
    print(
        f"  {m.title} ({m.released})  [type={type(m).__name__}, label attr exists={hasattr(type(m), '__label__')}]"
    )

print()
print("Option B — DTO output:")
for c in dto_results:
    # Consumer receives a MovieCard — no graph internals exposed.
    print(
        f"  {c.title} ({c.year})  [type={type(c).__name__}, label attr exists={hasattr(type(c), '__label__')}]"
    )

print()
print("NodeModel Output fields :", list(Movie.model_fields.keys()))
print("DTO     Output fields   :", list(MovieCard.model_fields.keys()))

The two outputs have the same data but different contracts:

| | Option A (`NodeModel`) | Option B (DTO) |
|-|------------------------|----------------|
| `Output` type | `Movie` — a graph declaration | `MovieCard` — a plain `BaseModel` |
| Graph internals visible to consumer | Yes (`__label__`, `__uid_field__`, …) | No |
| Field rename independence | No — stored and API name are the same | Yes — `released` → `year` absorbed in `materialize` |
| Schema change blast radius | Any change to `Movie` hits all queries that return it | Only this query's `materialize` needs updating |
| Verbosity | Less — `model_validate(raw)` shortcut | More — explicit field mapping |
| Right when | 1:1 full-node fetch, no API/storage divergence | Any rename, hidden field, joined data, or independent versioning |

**Practical rule:** start with Option A if the query truly returns a whole node
unchanged. Switch to Option B the first time the API contract diverges from the
stored shape — or from the start if you want a strict consumer boundary.

---
## Validating queries against the graph model

Orthograph provides two levels of static validation — no database connection
required:

| Function | What it checks | Import |
|----------|---------------|--------|
| `validate_cypher(template, graph_definition)` | One query string against the model | `orthograph.cypher.parser` |
| `validate_query_catalogue(catalogue, graph_definition)` | All registered queries at once | `orthograph.cypher.validation` |

Both return a `ValidationResult`. The result is *valid* (`is_valid=True`) when
there are zero ERROR-severity issues. INFO issues (unverifiable, alignment hints)
never affect validity.

**Checks performed for declarative Cypher queries:**

| Issue code | Severity | Meaning |
|-----------|----------|---------|
| `QUERY_UNKNOWN_NODE_LABEL` | ERROR | Label not declared in model |
| `QUERY_UNKNOWN_REL_TYPE` | ERROR | Relationship type not declared |
| `QUERY_UNKNOWN_PROPERTY` | ERROR | Property key not on the model type |
| `QUERY_INVALID_ENDPOINT` | ERROR | `(A)-[R]->(B)` contradicts declared endpoints |
| `QUERY_RETURN_OUTPUT_MISMATCH` | INFO | `Output` field has no matching RETURN column |
| `QUERY_USES_IDENTIFIER_INJECTION` | INFO | Template contains `<<name>>` placeholders |
| `QUERY_UNVERIFIABLE` | INFO | Query cannot be statically checked (imperative or identifier-injection) |

In [ ]:
from orthograph.cypher.parser import validate_cypher
from orthograph.cypher.validation import validate_query_catalogue
from orthograph.query.catalogue import QueryCatalogue


def show_result(label: str, result) -> None:
    """Print a compact summary of a ValidationResult."""
    status = "VALID" if result.is_valid else "INVALID"
    print(f"{label}: {status}")
    for issue in result.issues:
        print(f"  {issue}")
    if not result.issues:
        print("  (no issues)")

### Validating Shapes 1–3 as a catalogue

Shapes 1, 2, and 3 are all **declarative** queries (they have a `cypher_template`
and use only known labels, relationship types, and properties). They should all
validate cleanly.

Shape 4 (path query) is **imperative** — no `cypher_template` — so it will
produce a `QUERY_UNVERIFIABLE` INFO issue.

In [ ]:
# Register the queries defined in the earlier sections.
catalogue = QueryCatalogue()
catalogue.register_read(AllMovies())
catalogue.register_read(ActorMoviePairs())
catalogue.register_read(ActorFilmRows())
catalogue.register_read(ActorMovieEdges())
catalogue.register_read(ShortestActorPath())  # imperative — Shape 4

result = validate_query_catalogue(catalogue, graph_definition)
show_result("Full catalogue", result)

**Reading the output:**

- `is_valid=True` — INFO issues never affect validity, only ERRORs do.
- `shortest_actor_path` produces `QUERY_UNVERIFIABLE` because it is imperative
  (no `cypher_template`). The validator never silently skips a query — it always
  reports *why* it could not check.
- The remaining `QUERY_RETURN_OUTPUT_MISMATCH` INFO issues come from queries that
  return whole node variables (`RETURN m`, `RETURN p, m`, `RETURN p, r, m`)
  but declare an `Output` model with differently-named fields.

**Why the RETURN→Output mismatch INFO for whole-node returns?**

The alignment check compares the projected *column names* from the `RETURN`
clause against the *field names* of the `Output` model:

- `RETURN m` → graphglot sees column `m`; `Output = Movie` has fields `title`,
  `released`, `tagline`. Names don't match → INFO for each missing field.
- `RETURN p, m` → columns `p`, `m`; `Output = ActorMoviePair` has fields
  `person`, `movie`. Same naming gap → INFO.

These are **expected and intentional**: `materialize` bridges the naming gap.
The flat scalar projection (`ActorFilmRows` — `RETURN p.name AS actor_name, ...`)
produces *no* mismatch INFO because the `AS` aliases match the `Output` fields
exactly. That is the recommended pattern when you want validation to be silent.

### Single-query validation with `validate_cypher`

For quick checks outside a catalogue, call `validate_cypher` directly on a
template string. The flat-projection query (Shape 2 scalar variant) is a good
example: all labels and properties are known, so it should be valid.

In [ ]:
# Shape 2 flat-projection template — all identifiers are in the model.
flat_template = (
    "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) "
    "RETURN p.name AS actor_name, m.title AS movie_title, m.released AS released"
)
show_result("Flat projection", validate_cypher(flat_template, graph_definition))

# Shape 3 template — node + relationship + node
triple_template = "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p, r, m"
show_result("Triple (p,r,m)", validate_cypher(triple_template, graph_definition))

### Catching errors — unknown labels, types, properties, and endpoint violations

The four ERROR codes are caught statically. Each example below triggers one.

In [ ]:
# 1. Unknown node label
show_result(
    "Unknown label",
    validate_cypher(
        "MATCH (s:Studio)-[:MADE]->(m:Movie) RETURN s, m",
        graph_definition,
    ),
)
print()

# 2. Unknown relationship type
show_result(
    "Unknown rel type",
    validate_cypher(
        "MATCH (p:Person)-[:DIRECTED]->(m:Movie) RETURN p, m",
        graph_definition,
    ),
)
print()

# 3. Unknown property
show_result(
    "Unknown property",
    validate_cypher(
        "MATCH (m:Movie) RETURN m.budget",  # 'budget' not in Movie model
        graph_definition,
    ),
)
print()

# 4. Endpoint violation — ACTED_IN is declared Person→Movie, not Movie→Person
show_result(
    "Invalid endpoint",
    validate_cypher(
        "MATCH (m:Movie)-[r:ACTED_IN]->(p:Person) RETURN m, r, p",
        graph_definition,
    ),
)

### What the validator cannot check

Three situations produce `QUERY_UNVERIFIABLE` INFO instead of a definitive pass/fail:

**1. Imperative queries** (no `cypher_template`) — the Cypher is only known at
`build()` time. Shape 4 (path query) falls here. The executor still runs a
runtime syntax check via `parse_cypher` before hitting the driver, but label/
property/endpoint correctness is not checked statically.

**2. Identifier-injection queries** (`<<name>>` placeholders) — the static
template is incomplete without the runtime identifier values. The validator
emits `QUERY_USES_IDENTIFIER_INJECTION` and `QUERY_UNVERIFIABLE` and stops.

**3. Variable-length path patterns** — these are caught at class-definition time
by `_validate_declarative_cypher`, which forces you to use the imperative escape
hatch (and therefore also land in case 1 above).

In [ ]:
import warnings

from pydantic import BaseModel as _BM


# --- Identifier-injection query ---
# Queries the model with a runtime-substituted label.
# The template uses <<label>> which is not valid Cypher until substituted.
class LabelIdentifiers(_BM):
    label: str


class NodesByLabelParams(_BM):
    pass


class NodesByLabel(CypherReadQuery[NodesByLabelParams, Movie]):
    """Find nodes by a runtime-injected label."""

    name = "nodes_by_label"
    Identifiers = LabelIdentifiers
    cypher_template = "MATCH (n:<<label>>) RETURN n"

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie.model_validate(raw["n"])


inj_catalogue = QueryCatalogue()
inj_catalogue.register_read(NodesByLabel(identifiers={"label": "Movie"}))

result_inj = validate_query_catalogue(inj_catalogue, graph_definition)
show_result("Identifier injection", result_inj)

In [ ]:
# --- Path query (imperative) ---
# ShortestActorPath was already defined above. Registering it alone:
path_catalogue = QueryCatalogue()
path_catalogue.register_read(ShortestActorPath())

result_path = validate_query_catalogue(path_catalogue, graph_definition)
show_result("Path query (imperative)", result_path)

### Validation summary across all four shapes

| Shape | Query style | `validate_query_catalogue` outcome |
|-------|-------------|------------------------------------|
| 1 — Single node (`RETURN m`) | Declarative | VALID — labels, rels, properties checked; INFO for `m` vs `Output` field names |
| 2 — Multiple nodes (`RETURN p, m`) | Declarative | VALID — endpoint check passes; INFO for variable-name vs Output field-name gap |
| 2 — Scalar projection (`RETURN p.name AS actor_name, ...`) | Declarative | VALID — no INFO (AS aliases match Output fields exactly) |
| 3 — Node + rel + node (`RETURN p, r, m`) | Declarative | VALID — endpoint direction verified; INFO for variable-name vs Output field-name gap |
| 4 — Path (`RETURN path`) | Imperative | QUERY_UNVERIFIABLE INFO — no static check possible |

---
## Known gaps and future work

| Gap | Status | Tracking |
|-----|--------|----------|
| `materialize` is mandatory boilerplate for 1:1 column→field mapping | Design decision pending | E33 Q1, ADR-025 |
| Write queries discard `RETURN` rows (cannot echo created node) | Design decision pending | E33 Q2, ADR-026 |
| Variable-length path patterns (`*1..3`) rejected by graphglot | Tech debt | E20 |
| No built-in `PathOutput` wrapper type | Not planned for v0.1 | — |